<a href="https://colab.research.google.com/github/WhoisMonesh/Colab-Mega-Downloader/blob/main/Colab-Mega-Downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mega Downloader

Download files from Mega.nz links directly to Google Drive.


### 1. Mount Google Drive

In [0]:
from google.colab import drive
drive.mount('/content/drive')

### 2. Install Dependencies

In [0]:
!pip install mega.py -q
print('Dependencies installed.')

### 3. Configuration

In [0]:
# === CONFIGURATION ===
SAVE_PATH = '/content/downloads/MegaDownloader/'  # local temp dir
DRIVE_PATH = '/content/drive/My Drive/MegaDownloader/'  # final destination

MEGA_LINK = ""  # Mega.nz file/folder link
MEGA_EMAIL = ""  # Mega account email (optional)
MEGA_PASSWORD = ""  # Mega account password

KEEP_ALIVE = True  # prevent Colab timeout

# === END CONFIGURATION ===

### 4. Download

In [0]:
import os, time, datetime, shutil, glob
from google.colab import files

def format_bytes(n):
    for u in ['B', 'KB', 'MB', 'GB', 'TB']:
        if n < 1024: return f'{n:.1f} {u}'
        n /= 1024
    return f'{n:.1f} PB'
def main():
    os.makedirs(SAVE_PATH, exist_ok=True)
    print(f'Save path: {SAVE_PATH} (local)')
    if KEEP_ALIVE:
        from IPython.display import display, Javascript
        display(Javascript("""function keepAlive(){var btn=document.querySelector(\"colab-connect-button\");if(btn)btn.click()}setInterval(keepAlive,120000);"""))
        print('Keep-alive active')
    begin = time.time()
    from IPython.display import display, HTML
    progress_display = display(HTML('<pre>Starting...</pre>'), display_id='dl-progress')

    from mega import Mega
    def download_mega(link, save_path, email, password):
        m = Mega().login(email, password) if email and password else Mega().login()
        print("Logged into Mega")
        result = m.download_url(link, save_path)
        print("Download complete")
        return result
    print("Downloading from Mega...")
    download_mega(MEGA_LINK, SAVE_PATH, MEGA_EMAIL, MEGA_PASSWORD)
    end = time.time()
    elapsed = end - begin
    print(f'\n{"="*50}')
    print('COMPLETE')
    print(f'Elapsed: {int(elapsed//60)}m {int(elapsed%60)}s')
    print(f'Saved locally: {SAVE_PATH}')

    items = glob.glob(os.path.join(SAVE_PATH, '*'))
    items = [f for f in items if os.path.isfile(f)]
    if len(items) > 1:
        import zipfile
        total = sum(os.path.getsize(f) for f in items)
        processed = 0
        zpath = SAVE_PATH.rstrip('/').rstrip('\\') + '.zip'
        print(f'\nZipping {len(items)} files...')
        with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as zf:
            for fpath in items:
                zf.write(fpath, os.path.basename(fpath))
                processed += os.path.getsize(fpath)
                pct = int(processed * 100 / total) if total else 0
                bar = '#' * (pct // 2) + '-' * (50 - pct // 2)
                progress_display.update(HTML(f'<pre>Zipping: |{bar}| {pct}% | {os.path.basename(fpath)}</pre>'))
        print(f'Zip: {zpath}')
        print(f'Size: {format_bytes(os.path.getsize(zpath))}')
        display(HTML(f'<a href="{zpath}" download>Download ZIP</a>'))
        files.download(zpath)

    print(f'\nMoving to Drive...')
    os.makedirs(DRIVE_PATH, exist_ok=True)
    for f in os.listdir(SAVE_PATH):
        shutil.move(os.path.join(SAVE_PATH, f), os.path.join(DRIVE_PATH, f))
    print(f'Final: {DRIVE_PATH}')
    print(f'{"="*50}')
if __name__ == '__main__':
    main()
